## Sample Statistical visualization
- Color: #326a81
    - Control: '#aaadb2'
    - CD: '#9fbcca'
    - UC: '#326a81'
    - PSC+UC: '#7876b1c0'

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats
from scipy.stats import kruskal
from scipy.stats import mannwhitneyu
import math
from functools import lru_cache


# Total read Check(Rarefaction) & genera counts for observed per individual

These values were analyzed for samples with exclusion criteria applied.

- Rarefaction: 6284
- Feature counts: 814 (not prevalence < 10% filtered)
- Mean genera counts: 77.05
- After prevalence < 10% filtered feature counts: 193
- Mean genera counts: 65.43

In [ ]:
## Before prevalence filtering

te2 = pd.read_csv("data/2_preprocessed/ft.tsv", sep='\t')

print(te2["SRR23478191"].sum())
print(te2["SRR23478198"].sum())

genera = te2.shape[0]

print(f"feature counts:{genera}")

counts = te2.drop(columns=["#OTU ID"])

genera_per_sample = (counts > 0).sum(axis=0)

mean_genera = genera_per_sample.mean()

print(f"mean genera counts: {mean_genera.round(2)}")

6279.0
6279.0
feature counts:756
mean genera counts: 78.52


In [ ]:
## After prevalence filtering

te3 = pd.read_csv("data/2_preprocessed/prv10/prv10_ft.tsv", sep='\t', index_col=0)

te3_T = te3.T

print(te3_T["SRR23478191"].sum())
print(te3_T["SRR23478198"].sum())

genera = te3_T.shape[0]

print(f"feature counts:{genera}")

counts = te3_T

genera_per_sample = (counts > 0).sum(axis=0)

mean_genera = genera_per_sample.mean()

print(f"mean genera counts: {mean_genera.round(2)}")

6245.0
6272.0
feature counts:189
mean genera counts: 65.09


# Metadata statistical test

## Before apply to exclusion criteria

In [4]:
meta = pd.read_csv("data/pre_metadata.tsv", sep='\t')

In [5]:
co = meta.loc[meta['Host_disease'] == 'Control'].copy()
uc = meta.loc[meta['Host_disease'] == 'ulcerative colitis'].copy()
cd = meta.loc[meta['Host_disease'] == "crohn's disease"].copy()

In [6]:
## Age

co["Age"] = co["Months"] / 12
co["Age"] = co["Age"] - 0.5

uc["Age"] = uc["Months"] / 12
uc["Age"] = uc["Age"] - 0.5

cd["Age"] = cd["Months"] / 12
cd["Age"] = cd["Age"] - 0.5




| Host factors            | Control   | CD         | UC               |
| ----------------------- | --------- | ---------- | -----------------|
| Count                   | 95        | 21         | 41               |
| AGE(Mean (SD))          | 7.45(4.18)| 12.92(2.69)|  11.63(4.33)     |
| AGE range               |           |            |                  |
| Infants(0 ~ 3 age)      | 23        | 0          | 3                |
| Toddler(4 ~ 6 age)      | 15        | 0          | 3                |
| School-aged(7 ~ 11 age) | 40        | 4          | 6                |
| Teenager(12 ~ 18 age)   | 16        | 8          | 25               |
| none                    | 0         | 8          | 3                |
| BMI(MEAN (SD))          | none      | none       | 18.91(4.33)(n=27)|
| Sex                     |           |            |                  |
| Male                    | 48        | 14         | 3                |
| Female                  | 47        | 7          | 11               |
| Unknown                 | 0         | 0          | 27               |

In [7]:
meta['Host_disease'].value_counts(dropna=False)

Host_disease
Control               95
ulcerative colitis    38
crohn's disease       13
Name: count, dtype: int64

In [8]:
dfs = {"co": co, "cd": cd, "uc": uc}


rows = []

for name, df in dfs.items():
    count = df["#SampleID"].count()
    mean = df["Age"].mean()
    std = df["Age"].std()
    sex = df["host_sex"].value_counts(dropna=False)
    bmi = df["BMI"].mean()
    bmi_std = df["BMI"].std()
    
    rows.append({
        "Group": name,
        "n": count,
        "Age": f"{mean:.2f} ({std:.2f})",
        "BMI": f"{bmi:.2f} ({bmi_std:.2f})",
        "Sex": sex.to_dict()
    })

table1 = pd.DataFrame(rows)


table1

,Group,n,Age,BMI,Sex
0,co,95,7.45 (4.18),nan (nan),"{'male': 48, 'female': 47}"
1,cd,13,12.92 (2.69),nan (nan),"{'male': 8, 'female': 5}"
2,uc,38,11.63 (4.33),18.91 (4.20),"{nan: 27, 'female': 9, 'male': 2}"


In [9]:
meta_bmi = meta[meta["BMI"].notna()]

meta_bmi['location_disease'].value_counts()

location_disease
PIBD(IT)    27
Name: count, dtype: int64

## After QIIME2 rarefaction. Applied sample exclusion criteria

In [10]:
meta_aft = meta[(meta["#SampleID"] != "SRR23478219") & (meta["#SampleID"] != "SRR23492541") & (meta["#SampleID"] != "SRR29856068") & 
                (meta["Host_disease"] != "Primary sclerosing cholangitis and ulcerative colitis")]

## rarefaction, patients(PSC+UC) exclude

## If not included age information, droped.
meta_aft = meta_aft[meta_aft["Months"].notna()]

In [11]:
meta_aft["Host_disease"].value_counts()

Host_disease
Control               94
ulcerative colitis    37
crohn's disease       12
Name: count, dtype: int64

In [ ]:
## Separation to after rarefaction information

co = meta_aft.loc[meta_aft['Host_disease'] == 'Control'].copy()
uc = meta_aft.loc[meta_aft['Host_disease'] == 'ulcerative colitis'].copy()
cd = meta_aft.loc[meta_aft['Host_disease'] == "crohn's disease"].copy()


## Age

co["Age"] = co["Months"] / 12
co["Age"] = co["Age"] - 0.5

uc["Age"] = uc["Months"] / 12
uc["Age"] = uc["Age"] - 0.5

cd["Age"] = cd["Months"] / 12
cd["Age"] = cd["Age"] - 0.5




dfs = {"co": co, "cd": cd, "uc": uc}


rows1 = []

for name, df in dfs.items():
    count = df["#SampleID"].count()
    mean = df["Age"].mean()
    std = df["Age"].std()
    sex = df["host_sex"].value_counts(dropna=False)
    bmi = df["BMI"].mean()
    bmi_std = df["BMI"].std()
    
    rows1.append({
        "Group": name,
        "n": count,
        "Age": f"{mean:.2f} ({std:.2f})",
        "BMI": f"{bmi:.2f} ({bmi_std:.2f})",
        "Sex": sex.to_dict()
    })

table2 = pd.DataFrame(rows1)


table2.to_csv("data/3_results/1_statistical/aft_pre_statistical.csv", sep=',')

In [13]:
table2

,Group,n,Age,BMI,Sex
0,co,94,7.50 (4.17),nan (nan),"{'male': 48, 'female': 46}"
1,cd,12,12.75 (2.73),nan (nan),"{'male': 7, 'female': 5}"
2,uc,37,11.62 (4.39),18.90 (4.29),"{nan: 26, 'female': 9, 'male': 2}"


In [14]:
## Age Count create

def age_category(age):
    if 0 <= age <= 3:
        return "Infants"
    elif 4 <= age <= 6:
        return "Toddler"
    elif 7 <= age <= 11:
        return "School-aged"
    elif 12 <= age <= 17:
        return "Teenager"
    else:
        return "Unknown"

for df in [co, cd, uc]:
    df["range"] = df["Age"].apply(age_category)

In [15]:
co['range'].value_counts()

range
School-aged    40
Infants        23
Teenager       16
Toddler        15
Name: count, dtype: int64

| Host factors            | Control   | CD         | UC               |
| ----------------------- | --------- | ---------- | -----------------|
| Count                   | 94        | 12         | 37               |
| AGE(Mean (SD))          | 7.45(4.18)| 12.75(2.73)| 11.62(4.39)      |
| AGE range               |           |            |                  |
| Infants(0 ~ 3 age)      | 23        | 0          | 3                |
| Toddler(4 ~ 6 age)      | 15        | 0          | 3                |
| School-aged(7 ~ 11 age) | 40        | 4          | 6                |
| Teenager(12 ~ 18 age)   | 16        | 8          | 25               | 
| BMI(MEAN (SD))          | none      | none       | 18.90(4.26)(n=26)|
| Sex                     |           |            |                  | 
| Male                    | 48        | 7          | 2                | 
| Female                  | 46        | 5          | 9                | 
| Unknown                 | 0         | 0          | 26               | 

## Statistical test by each value

In [16]:
stat, p = kruskal(
    co["Age"].dropna(),
    uc["Age"].dropna(),
    cd["Age"].dropna()
)



print("Kruskal statistic: ", stat)
print("p-value", p)

Kruskal statistic:  31.039706938166418
p-value 1.8189186603935436e-07


In [17]:
def table_log_probability(table):
    """
    Hypergeometric probability under fixed margins.
    log P(table) = sum(row_i!) + sum(col_j!) - N! - sum(cell_ij!)
    """
    table = np.asarray(table, dtype=int)

    row_sums = table.sum(axis=1)
    col_sums = table.sum(axis=0)
    total = table.sum()

    logp = 0.0
    logp += sum(math.lgamma(x + 1) for x in row_sums)
    logp += sum(math.lgamma(x + 1) for x in col_sums)
    logp -= math.lgamma(total + 1)
    logp -= sum(math.lgamma(x + 1) for x in table.ravel())

    return logp


def generate_tables(row_sums, col_sums):
    """
    Generate all contingency tables with fixed row and column sums.
    Works for small/moderate tables.
    """

    row_sums = tuple(row_sums)
    col_sums = tuple(col_sums)

    n_rows = len(row_sums)
    n_cols = len(col_sums)

    table = np.zeros((n_rows, n_cols), dtype=int)

    def rec(i, remaining_cols):
        remaining_cols = list(remaining_cols)

        if i == n_rows - 1:
            if sum(remaining_cols) == row_sums[i]:
                table[i, :] = remaining_cols
                yield table.copy()
            return

        row_total = row_sums[i]

        def fill_row(j, remaining_row, current_row, cols_left):
            if j == n_cols - 1:
                val = remaining_row
                if val <= cols_left[j]:
                    current_row[j] = val
                    new_cols = cols_left.copy()
                    new_cols[j] -= val

                    table[i, :] = current_row
                    yield from rec(i + 1, new_cols)
                return

            max_val = min(remaining_row, cols_left[j])

            for val in range(max_val + 1):
                current_row[j] = val
                new_cols = cols_left.copy()
                new_cols[j] -= val
                yield from fill_row(
                    j + 1,
                    remaining_row - val,
                    current_row.copy(),
                    new_cols
                )

        yield from fill_row(
            0,
            row_total,
            np.zeros(n_cols, dtype=int),
            remaining_cols
        )

    yield from rec(0, col_sums)


def fisher_freeman_halton_exact(table, tol=1e-12):
    """
    Exact Fisher-Freeman-Halton test for RxC table.
    Two-sided p-value: sum of probabilities of tables
    with probability <= observed table probability.
    """

    table = np.asarray(table, dtype=int)

    row_sums = table.sum(axis=1)
    col_sums = table.sum(axis=0)

    obs_logp = table_log_probability(table)

    p_value = 0.0
    n_tables = 0

    for candidate in generate_tables(row_sums, col_sums):
        logp = table_log_probability(candidate)
        prob = math.exp(logp)

        if logp <= obs_logp + tol:
            p_value += prob

        n_tables += 1

    return {
        "p_value": p_value,
        "observed_log_probability": obs_logp,
        "n_enumerated_tables": n_tables
    }

## Age
age_range_table = np.array([
    [23, 15, 40, 16],  # Control
    [0,  0,  4,  8],   # CD
    [3,  3,  6,  25]   # UC
])

result_age = fisher_freeman_halton_exact(age_range_table)


## Sex
sex_range_table = np.array([
    [48, 46, 0],
    [7, 5, 0],
    [2, 9, 26]
])

result_sex = fisher_freeman_halton_exact(sex_range_table)



In [18]:
print(result_age)
print(result_sex)

{'p_value': 4.549084064533483e-07, 'observed_log_probability': -26.255537807316557, 'n_enumerated_tables': 3255560}
{'p_value': 3.623362602883548e-20, 'observed_log_probability': -49.91882141315688, 'n_enumerated_tables': 56056}


| Host factors            | Control   | CD         | UC               | statistical test |
| ----------------------- | --------- | ---------- | -----------------|------------------|
| Count                   | 94        | 12         | 37               |                  |
| AGE(Mean (SD))          | 7.45(4.18)| 12.75(2.73)| 11.62(4.39)      | 1.8e-7(Kruskal wallis)|
| AGE range               |           |            |                  | 4.5e-7(Fisher's exact)|
| Infants(0 ~ 3 age)      | 23        | 0          | 3                | 
| Toddler(4 ~ 6 age)      | 15        | 0          | 3                |
| School-aged(7 ~ 11 age) | 40        | 4          | 6                |
| Teenager(12 ~ 18 age)   | 16        | 8          | 25               | 
| Sex                     |           |            |                  | 3.6e-20(Fisher's exact)|
| Male                    | 48        | 7          | 2                | 
| Female                  | 46        | 5          | 9                | 
| Unknown                 | 0         | 0          | 26               | 